# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
# Step 1: Completely remove all the conflicting packages
# so we start from a clean slate with no cached builds
!pip uninstall -y transformers huggingface_hub sentence-transformers \
    datasets tokenizers -q

In [2]:
# Step 2: Reinstall the entire stack together in one command.
# Installing them together forces pip to resolve a mutually
# compatible set of versions rather than installing them
# sequentially and creating conflicts
!pip install \
    "huggingface_hub==0.21.4" \
    "transformers==4.38.2" \
    "sentence-transformers==2.6.1" \
    "datasets==2.14.7" \
    "tokenizers==0.15.2" \
    -q

print("✅ Done. NOW restart the runtime before running any other cell.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires accelerate>=0.21.0, which is not installed.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.21.4 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.21.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.21.4 which is incompatible.
✅ Done. NOW restart the runtime before running any other cell.


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [5]:
# load the dataset from huggingface in streaming mode and shuffle it
from datasets import load_dataset
wiki_data = load_dataset(
    'vblagoje/wikipedia_snippets_streamed',
    split='train',
    streaming=True
).shuffle(seed=960)

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [6]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'wiki_id': 'Q7649565',
 'start_paragraph': 20,
 'start_character': 272,
 'end_paragraph': 24,
 'end_character': 380,
 'article_title': 'Sustainable Agriculture Research and Education',
 'section_title': "2000s & Evaluation of the program's effectiveness",
 'passage_text': "preserving the surrounding prairies. It ran until March 31, 2001.\nIn 2008, SARE celebrated its 20th anniversary. To that date, the program had funded 3,700 projects and was operating with an annual budget of approximately $19 million. Evaluation of the program's effectiveness As of 2008, 64% of farmers who had received SARE grants stated that they had been able to earn increased profits as a result of the funding they received and utilization of sustainable agriculture methods. Additionally, 79% of grantees said that they had experienced a significant improvement in soil quality though the environmentally friendly, sustainable methods that they were"}

In [7]:
# The 'wiki_snippets' dataset does not have 'section_title', so we will proceed without this specific filter
history = wiki_data.filter(
    lambda d: "History" in (d['section_title'] or "")
)

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [9]:
from tqdm.auto import tqdm  # progress bar

total_doc_count = 10000 # you can consider 10000 also

counter = 0
docs = []
# iterate through the dataset and apply our filter
for d in tqdm(history, total=total_doc_count):
# extract the fields we need - article, section, and passage
  article_title = d['article_title']
  section_title = d['section_title']
  passage_text = d['passage_text']
# package them together as a dict and append to our docs list
  docs.append({
      "article_title": article_title,
      "section_title": section_title,
      "passage_text": passage_text
  })
  # increase the counter on every iteration
  counter += 1
  if counter == total_doc_count:
    break

  0%|          | 0/10000 [00:00<?, ?it/s]

In [10]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Taupo District,History,was not until the 1950s that the region starte...
1,Sutarfeni,History & Western asian analogues,Sutarfeni History strand-like pheni were Phena...
2,The Bishop Wand Church of England School,History,The Bishop Wand Church of England School Histo...
3,Teufelsmoor,History & Situation today,"made to preserve the original landscape, altho..."
4,Surface Hill Uniting Church,History,in perpetual reminder that work and worship go...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [11]:
from google.colab import userdata
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec

# initialize connection to pinecone (get API key at app.pinecone.io)
pinecone_api_key = os.environ.get('PINECONE_API_KEY') or userdata.get('PINECONE_API_KEY')


Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [12]:
spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [13]:
index_name = 'abstractive-question-answering'

import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
  # if it doesn't exist, create it with the right dimensions and metric
    pc.create_index(
        name=index_name,
        dimension=768,
        metric='cosine',
        spec=spec
    )
  # wait for the index to be ready before we try to use it
    while not pc.describe_index(index_name).status['ready']:
      time.sleep(1)
# connect to the index so we can interact with it
index = pc.Index(index_name)

# verify the index is empty (stats should show 0 vectors)
print(index.describe_index_stats())

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [14]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device) #load the retriever model from HuggingFace. Use the flax-sentence-embeddings/all_datasets_v3_mpnet-base model
retriever

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: MPNetModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [15]:
# we will use batches of 64
batch_size = 64

#You will create embedding for the passage_text variable and be use to include the meta data in each batch
for i in tqdm(range(0, len(df), batch_size)):

    # find end of batch - this handles the last batch which may be smaller than 64
    i_end = min(i + batch_size, len(df))

    # extract batch - slice the dataframe from current position to end of batch
    batch = df.iloc[i:i_end]

    # generate embeddings for passage_text column of this batch
    # .tolist() converts the panda Series to a plain Python list, which the Retriever expects
    embeds = retriever.encode(batch['passage_text'].tolist()).tolist()

    # build the metadata list - one dict per document in the batch
    # this is what gets stored alongside the vector in Pinecone
    metadata = [
        {
            'article_title': row['article_title'],
            'section_title': row['section_title'],
            'passage_text': row['passage_text']
        }
        for _, row in batch.iterrows()
    ]

    # create the list of (id, vector, metadata) tuples to upsert
    # the id just needs to be a unique string - we use the dataframe index
    vectors = [
        (str(i + j), embeds[j], metadata[j])
        for j in range(len(batch))
    ]

    # upsert this batch of vectors into Pinecone index
    index.upsert(vectors=vectors)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/157 [00:00<?, ?it/s]

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 10000}},
 'total_vector_count': 10000}

## What we have so far

It's worth pausing to appreciate the architecture that's now in place, because we've assembled something genuinely sophisticated. We have a **semantic search engine** backed by a vector database. Every one of our 10,000 passages now lives as a coordinate in a 768-dimensional mathematical space, positioned relative to every other passage based on meaning rather than keywords.

This is fundamentally different from traditional search — a keyword search for "Napoleon's defeat" would only find documents containing those exact words, whereas your system will find documents that are semantically related to that concept, even if they use entirely different vocabulary like "Bonaparte's final campaign" or "the Battle of Waterloo's outcome."

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [16]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [18]:
def query_pinecone(query, top_k):
    # convert the query string into a 768-dim vector using
    # the same retriever model we used to embed the passages — consistency here is critical
    # both query and passages must live in the same vector space

    xq = retriever.encode([query]).tolist()

    # search Pinecone for the top_k most semantically similar passage vectors
    # include_metadata=True ensures we get the passage text back, not just IDs

    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)

    return xc

In [19]:
def format_query(query, context):
    # wrap each retrieved passage in a <P> tag — this is the format
    # the ELI5 BART model was specifically fine-tuned on, so it expects
    # passages to be delimited this way in order to generate good answers
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]

    # join all context passages into a single string separated by a space
    # this gives BART one continuous block of evidence to reason over
    context = " ".join(context)

    # prepend the question to the context with a double newline separator,
    # which is the input format the ELI5 model was trained to expect
    query = f"question: {query} context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [20]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

{'matches': [{'id': '3906',
              'metadata': {'article_title': 'Electric power system',
                           'passage_text': 'Electric power system History In '
                                           '1881, two electricians built the '
                                           "world's first power system at "
                                           'Godalming in England. It was '
                                           'powered by two waterwheels and '
                                           'produced an alternating current '
                                           'that in turn supplied seven '
                                           'Siemens arc lamps at 250 volts and '
                                           '34 incandescent lamps at 40 volts. '
                                           'However, supply to the lamps was '
                                           'intermittent and in 1882 Thomas '
                                           'Ed

In [21]:
from pprint import pprint

In [22]:
# format the query in the form generator expects the input
query = format_query(query, result['matches'])
pprint(query)

('question: when was the first electric power system built? context: <P> '
 "Electric power system History In 1881, two electricians built the world's "
 'first power system at Godalming in England. It was powered by two '
 'waterwheels and produced an alternating current that in turn supplied seven '
 'Siemens arc lamps at 250 volts and 34 incandescent lamps at 40 volts. '
 'However, supply to the lamps was intermittent and in 1882 Thomas Edison and '
 'his company, The Edison Electric Light Company, developed the first '
 'steam-powered electric power station on Pearl Street in New York City. The '
 'Pearl Street Station initially powered around 3,000 lamps for 59 customers. '
 'The power station generated direct current and')


The output looks great. Now let's write a function to generate answers.

In [32]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, truncation=True, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [33]:
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of the US '
 'government. It cost about $100 billion to build.')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [34]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent by a telegraph. The first telegraph was '
 'built in the early 1800s. The first telegraph was built in the late 1800s. '
 'The first te')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [26]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

audio distance records, and were heard as far west as Hawaii. They were also received in Paris, France, which marked the first transmission of speech across the Atlantic.
With the entrance of the United States into World War I in April 1917 the federal government took over full control of the radio industry, and it became illegal for civilians to possess an operational radio receiver. However NAA continued to operate during the conflict. In addition to time signals and weather reports, it also broadcast news summaries received by troops on land and aboard ships in the Atlantic. Effective April 15, 1919
---
radio operators, interested in a practical benefit from their hobby, and jewelers, who previously had been reliant on time services transmitted over telegraph wires, which had a reputation for being both expensive and of questionable reliability, especially compared to the free and very accurate NAA transmissions.
NAA's original transmitters were only capable of producing the dots-an

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [27]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 is not a virus. It is a bacterium. It is not a virus. It is a '
 'bacterium. It is a bacterium. It is a bacterium')


In [28]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

diseases when they occupy at certain times of the year natural habitat of a certain pathogen (plague, tularemia, leptospirosis, arboviruses, tick-borne relapsing fever. The WHO Expert Committee on Zoonoses listed over 100 such diseases. About natural focality of the diseases is known elsewhere. History Historically, Sanitary epidemiological reconnaissance implied collection and transfer of all data available on sanitary and epidemiological situation of the area of possible deployment and action of armed forces, the same data for the neighbouring and enemy armed forces. The aim for the reconnaissance was to clear up the reasons of the specific disease origin—sources of the
---
1974 during the Super Outbreak (one of seven F5 tornadoes during that outbreak) which killed three and demolished many homes.
---
stationary during wartime and emergency of peace time the sanitary epidemiological reconnaissance turns into sanitary and epidemiological surveillance and medical control of vital and c

Let’s finish with a final few questions.

In [29]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The War of Currents was a rivalry between Edison and Westinghouse over the '
 'use of alternating current in power transmission. The two companies were '
 'competing for the use of alternating current in power transmission')


In [30]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The first man to walk on the moon was Neil Armstrong in 1969. He walked on '
 'the moon in 1969. He was the first man to walk on the moon.')


In [31]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of the US '
 'government. It cost about $100 billion to build.')


As we can see, the model can generate some decent answers.

#### Add a few more questions

In [58]:
query = "What were the causes of the French Revolution?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)


('The French Revolution was a series of events that started in 1789, and ended '
 'in 1789. The events that led to the Revolution were: * The French Revolution '
 'was a series of events')


In [59]:
for doc in context["matches"]:
    print(f"Article : {doc['metadata']['article_title']}")
    print(f"Section : {doc['metadata']['section_title']}")
    print(f"Score   : {doc['score']:.4f}")   # cosine similarity, higher = more relevant
    print(f"Passage : {doc['metadata']['passage_text']}")
    print("---")

Article : Confédération générale de la production française
Section : History
Score   : 0.4483
Passage : the socialist experiment will succeed and it will be a matter of knowing what we can do to avoid passing completely under the absolute control of a totalitarian state, or this experiment will not succeed, and then, it will end up with bloodshed ... in either case, it will be necessary to reduce the damage to a minimum, I do not say avoid it.
In 7 June 1936 Alexandre Lambert-Ribot, secretary general of the Comité des forges, signed the Matignon Agreements to end the general strike that had ensued.
CGPF President René-Paul Duchemin signed on behalf of French employers.
Forces led by the
---
Article : Fraissinet-de-Lozère
Section : History
Score   : 0.4332
Passage : the main characters of the Rouvière family, one of the most powerful groups of the village. This could mean that "newly converted" dit not plainly support their king.
During the War of the Camisards, it was very close to th

In [56]:
query = "How did the Roman Empire fall?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

("I'm not a historian, but I've read a lot about the fall of the Roman Empire, "
 'and I think I can answer your question. The fall of the Roman Empire was not '
 'a')


In [57]:
for doc in context["matches"]:
    print(f"Article : {doc['metadata']['article_title']}")
    print(f"Section : {doc['metadata']['section_title']}")
    print(f"Score   : {doc['score']:.4f}")   # cosine similarity, higher = more relevant
    print(f"Passage : {doc['metadata']['passage_text']}")
    print("---")

Article : Flevum
Section : History
Score   : 0.4308
Passage : destruction of the first castrum around 28 AD, but has left few archaeological evidences. It seems to have survived only until 55 AD.
---
Article : Iucundiana
Section : History
Score   : 0.4134
Passage : Carthage in 411, confronting Catholic (prevailing) and Donatist (condemned) bishops of Roman Africa.
---
Article : Diocese of Hispania
Section : History
Score   : 0.3851
Passage : of Hispania by the Vandals, Alans, and Suebi in 409 the diocese came to an end.
---


### Questions where it might hallucinate due to lack of context

In [52]:
query = "What is the current GDP of Germany?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The GDP of Germany is the sum of all the goods and services produced by the '
 'German economy. The GDP of Germany is the sum of all the goods and services '
 'produced by the German economy.')


In [53]:
for doc in context["matches"]:
    print(f"Article : {doc['metadata']['article_title']}")
    print(f"Section : {doc['metadata']['section_title']}")
    print(f"Score   : {doc['score']:.4f}")   # cosine similarity, higher = more relevant
    print(f"Passage : {doc['metadata']['passage_text']}")
    print("---")

Article : Mönchpfiffel-Nikolausrieth
Section : History
Score   : 0.3901
Passage : Saxony.
---
Article : Frauenkappelen
Section : History & Geography
Score   : 0.3462
Passage : of Bern. Geography Frauenkappelen has an area of 9.27 km² (3.58 sq mi). As of 2012, a total of 3.92 km² (1.51 sq mi) or 42.3% is used for agricultural purposes, while 3.81 km² (1.47 sq mi) or 41.1% is forested. Of the rest of the land, 0.63 km² (0.24 sq mi) or 6.8% is settled (buildings or roads), 0.86 km² (0.33 sq mi) or 9.3% is either rivers or lakes and 0.07 km² (17 acres) or 0.8% is unproductive land.
During the same year, housing and buildings made up 3.5% and transportation infrastructure made up 2.7%. Out of the forested land, all of the forested land area is covered with heavy forests. Of the agricultural land,
---
Article : Dubuque, Iowa
Section : History
Score   : 0.3451
Passage : a series of changes in manufacturing and the onset of the "Farm Crisis" led to a large decline in the sector and the city's 

In [54]:
query = "How do I fix a dependency conflict in Python?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

("I'm not sure if this is what you're looking for, but I can tell you that "
 "it's not a dependency conflict. It's a conflict between the two parties. The "
 'two parties')


In [55]:
for doc in context["matches"]:
    print(f"Article : {doc['metadata']['article_title']}")
    print(f"Section : {doc['metadata']['section_title']}")
    print(f"Score   : {doc['score']:.4f}")   # cosine similarity, higher = more relevant
    print(f"Passage : {doc['metadata']['passage_text']}")
    print("---")

Article : Lens Express
Section : History & Launch
Score   : 0.1572
Passage : also mentioned that certain states, including Hawaii, West Virginia, North Carolina, and Minnesota had restraint of trade laws in effect to prevent advertising for contact lenses. Launch The "way" for a mail order contact lens industry did not become clear until the Federal Trade Commission ruled in 1985, that eye doctors (Ophthalmologists and Optometrists) must provide the contact lens prescription to their patients so that the patient may shop for contact lenses as a consumer. Withholding a contact lens prescription would be considered restraint of trade.
Linda J. Kaplan, MD, Ophthalmologist, a consultant to the FTC, and a liaison to the
---
Article : Board for International Food and Agricultural Development (BIFAD)
Section : History & Challenges and strategies
Score   : 0.1364
Passage : US and the international scene. BIFAD was also tasked with the formulation of basic policy, procedures, and criteria for p

## Conclusion

In this notebook we built an end-to-end **Abstractive Question Answering (QA)** system
using a Retrieval-Augmented Generation (RAG) architecture. The pipeline combined three
core components: a **vector database** (Pinecone) to store and search semantic
representations of text, a **retriever** (MPNet SentenceTransformer) to convert both
passages and questions into 768-dimensional embeddings, and a **generator** (ELI5 BART)
to synthesise natural language answers from retrieved context.

### What We Built
- Loaded and filtered 10,000 Wikipedia history passages from a streaming HuggingFace dataset
- Generated semantic embeddings for every passage and indexed them in Pinecone using cosine similarity
- Implemented a query pipeline that retrieves the most relevant passages for any question and generates a fluent, abstractive answer

### Key Concepts Learned
- **RAG architecture**: separating retrieval (finding relevant evidence) from generation (articulating an answer)
- **Vector embeddings**: representing text as geometric coordinates where semantic similarity becomes a distance problem
- **Batched inference**: processing documents in groups of 64 to maximise GPU utilisation
- **Metadata storage**: attaching original text to vectors in Pinecone so retrieved results are human-readable

### Challenges Encountered
- **Dependency conflicts**: the Python ML ecosystem (HuggingFace, Pinecone, NumPy, transformers)
  has rapidly evolving APIs that frequently produce binary incompatibilities in shared environments
  like Colab — resolved by pinning interdependent packages to mutually compatible versions
- **Dataset loading scripts**: newer versions of the `datasets` library dropped support for
  custom loading scripts, requiring a downgrade to `datasets==2.14.7`
- **Pinecone API migration**: the notebook was written for Pinecone v3 (`pinecone.init()`),
  but the installed version used the newer v5+ object-oriented API (`Pinecone()` class)
- **Answer quality boundaries**: the system generates confidently wrong answers for
  out-of-domain questions, highlighting the need for relevance thresholds in production RAG systems

### Next Steps
Scaling to 50,000 documents, adding a re-ranking layer to improve retrieval precision,
and introducing a relevance threshold to handle unanswerable questions gracefully would
be natural extensions of this system.